# FingerBall v2 — Stage 2E：リリース前後の指先色分け動画

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2E_KeyTips_Showcase.ipynb)

**目的**：195〜200フレームを拡大し、**母指先端・示指先端・中指先端**を色分けした通常速度・スローモーション動画を出力します。Stage 2Dの多倍率Hand Landmarker・GrabCut参考マスク・21点CSV出力は維持。

**手動ROIはx0/y0のみ指定**し、そこから **200×200 px正方形**を既定値とします。前回Stage 2DのキーフレームROI左上座標を初期値に使用します。

重要：指先・手マスクはAIによる未検証の推定です。ボール位置・正確な離手タイミングは判定しません。共有前に映り込み・許諾を確認してください。

## 0. セットアップとGoogle Drive

In [ ]:
%pip -q install "mediapipe==0.10.35" "opencv-python-headless>=4.8" ipywidgets pandas matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定 — 基本的にここだけ変更

In [ ]:
from pathlib import Path
from google.colab import files, output
from IPython.display import display
import numpy as np, pandas as pd, cv2, mediapipe as mp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import urllib.request, shutil, json, zipfile
from datetime import datetime
output.enable_custom_widget_manager()

PROJECT_PATH = ''  # 例 '/content/drive/MyDrive/母指回内トラッキング'
VIDEO_PATH = ''    # 空なら手動アップロード
START_FRAME, END_FRAME = 175, 210
ANCHOR_FRAMES = (175, 190, 210)  # 3つのROIキーフレーム（解析区間内）
CROP_SCALE = 2.0     # 手のBOXを拡大してモデルに入力
HAND_MIN_DET = 0.15
HAND_MIN_PRESENCE = 0.2
HAND_MIN_TRACK = 0.2
RUN_GRABCUT = True
RELEASE_ZOOM = 3.0
OUTPUT_DIR_NAME = 'FingerBall_v2_stage2e_results'
SHOWCASE_START, SHOWCASE_END = 195, 200
SLOW_REPEAT = 4
ROI_SIZE = 200
DEFAULT_ANCHOR_TOPLEFT = {175:(726,155), 190:(660,88), 210:(702,231)}
RETRY_CROP_SCALES = (1.5, 2.0, 2.8, 3.6)
MIN_LANDMARK_VALID_RATIO = 0.85
MAX_PREV_CENTER_JUMP = 1.7  # ROI box side ratio. Larger movements are flagged, not silently hidden.
EXPORT_CLINICIAN_PACKAGE = True
TASK_URLS = [
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task',
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task']
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    roots=[Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]
    for root in roots:
        if (root/'母指回内トラッキング').is_dir():return root/'母指回内トラッキング'
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/'母指回内トラッキング').is_dir():return d/'母指回内トラッキング'
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画が指定されません')
    VIDEO_PATH='/content/'+next(iter(uploaded))
video_original=Path(VIDEO_PATH)
if not video_original.is_file():raise FileNotFoundError(video_original)
cache=Path('/content/fingerball_stage2c');cache.mkdir(exist_ok=True)
if str(video_original).startswith('/content/drive/'):
    video=cache/('input_'+video_original.name)
    if not video.exists() or video.stat().st_size!=video_original.stat().st_size:shutil.copy2(video_original,video)
else:video=video_original
cap=cv2.VideoCapture(str(video))
if not cap.isOpened():raise RuntimeError('動画が開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
assert 0<=START_FRAME<=END_FRAME<TOTAL
assert len(ANCHOR_FRAMES)>=2 and min(ANCHOR_FRAMES)>=START_FRAME and max(ANCHOR_FRAMES)<=END_FRAME
assert sorted(set(ANCHOR_FRAMES))==list(ANCHOR_FRAMES)
assert 0<=SHOWCASE_START<=SHOWCASE_END<TOTAL and START_FRAME<=SHOWCASE_START<=SHOWCASE_END<=END_FRAME
assert W>=ROI_SIZE and H>=ROI_SIZE
print('video:',video_original.name,'total:',TOTAL,'fps:',FPS,'size:',W,H,'range:',START_FRAME,END_FRAME)

## 2. ROIを指定：x0・y0スライダー、200×200 px正方形

代表フレーム175/190/210を選び、**x0/y0を調整するだけ**で正方形ROIを配置します。初期値は前回Stage 2Dの左上座標です。手動で変更したら「確定」をクリックしてください。未操作の場合も前回値で実行できます。

In [ ]:
anchor_boxes={}
def frame_at(index):
    cap=cv2.VideoCapture(str(video))
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(index))
    ok,im=cap.read();cap.release()
    if not ok: raise RuntimeError('Read frame '+str(index))
    return im
def square_box(x0,y0):
    x0=int(np.clip(x0,0,W-ROI_SIZE))
    y0=int(np.clip(y0,0,H-ROI_SIZE))
    return np.array([x0,y0,x0+ROI_SIZE,y0+ROI_SIZE],dtype=float)
frames_cache={i:frame_at(i) for i in ANCHOR_FRAMES}
chooser=widgets.Dropdown(options=list(ANCHOR_FRAMES),description='フレーム')
x0_slider=widgets.IntSlider(value=0,min=0,max=W-ROI_SIZE,description='x0',continuous_update=False,layout=widgets.Layout(width='95%'))
y0_slider=widgets.IntSlider(value=0,min=0,max=H-ROI_SIZE,description='y0',continuous_update=False,layout=widgets.Layout(width='95%'))
preview=widgets.Image(format='png')
confirm_btn=widgets.Button(description='このBOXで確定',button_style='success',icon='check')
msg=widgets.HTML()
def redraw(*args):
    img=frames_cache[int(chooser.value)].copy()
    b=square_box(x0_slider.value,y0_slider.value).astype(int)
    cv2.rectangle(img,tuple(b[:2]),tuple(b[2:]),(0,255,0),3)
    cv2.putText(img,f'ROI: {b[0]},{b[1]} / 200x200',(12,30),cv2.FONT_HERSHEY_SIMPLEX,.7,(0,255,0),2)
    scale=min(1.,960/W)
    if scale<1:img=cv2.resize(img,None,fx=scale,fy=scale)
    _,buf=cv2.imencode('.png',img)
    preview.value=buf.tobytes()
    msg.value='確定済み: '+', '.join(str(k) for k in sorted(anchor_boxes))
def change_frame(change):
    f=int(change['new'])
    b=anchor_boxes.get(f)
    x0,y0=(int(b[0]),int(b[1])) if b is not None else DEFAULT_ANCHOR_TOPLEFT[f]
    x0_slider.value=int(x0);y0_slider.value=int(y0)
    redraw()
def confirm(_):
    f=int(chooser.value)
    anchor_boxes[f]=square_box(x0_slider.value,y0_slider.value)
    redraw()
    print('確定:',f,anchor_boxes[f].astype(int).tolist())
chooser.observe(change_frame,names='value')
x0_slider.observe(redraw,names='value')
y0_slider.observe(redraw,names='value')
confirm_btn.on_click(confirm)
first=int(ANCHOR_FRAMES[0])
x0_slider.value=DEFAULT_ANCHOR_TOPLEFT[first][0]
y0_slider.value=DEFAULT_ANCHOR_TOPLEFT[first][1]
redraw()
display(widgets.VBox([chooser,preview,x0_slider,y0_slider,confirm_btn,msg]))
print('ROIは常に200x200 pxです。必要なら各フレームで位置を調整してください。')

## 3. ROIの補間プレビュー・MediaPipeモデルの取得

In [ ]:
if sorted(anchor_boxes)!=list(ANCHOR_FRAMES):
    missing=sorted(set(ANCHOR_FRAMES)-set(anchor_boxes))
    print('未確定フレームはStage 2D既定値を使用:',missing)
    for f in missing:
        anchor_boxes[f]=square_box(*DEFAULT_ANCHOR_TOPLEFT[f])
keys=np.array(sorted(anchor_boxes),dtype=int)
def interp_box(i):
    if i<=keys[0]:return anchor_boxes[int(keys[0])].copy()
    if i>=keys[-1]:return anchor_boxes[int(keys[-1])].copy()
    k=np.searchsorted(keys,i);a,b=int(keys[k-1]),int(keys[k])
    t=(i-a)/(b-a)
    return (1-t)*anchor_boxes[a]+t*anchor_boxes[b]
fig,axs=plt.subplots(1,len(ANCHOR_FRAMES),figsize=(15,5))
for ax,i in zip(np.ravel(axs),ANCHOR_FRAMES):
    img=frames_cache[i].copy();q=np.rint(interp_box(i)).astype(int)
    cv2.rectangle(img,tuple(q[:2]),tuple(q[2:]),(0,255,0),3)
    ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));ax.set_title(f'Frame {i}');ax.axis('off')
plt.show()
task_path=cache/'hand_landmarker.task'
if task_path.exists() and task_path.stat().st_size<1000000:task_path.unlink()
if not task_path.exists():
    errors=[]
    for url in TASK_URLS:
        tmp=cache/'hand_download.tmp'
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req,timeout=90) as response,open(tmp,'wb') as out:shutil.copyfileobj(response,out)
            if tmp.stat().st_size<1000000:raise RuntimeError('Model file too small')
            tmp.replace(task_path);break
        except Exception as e:
            errors.append(str(e));tmp.unlink(missing_ok=True)
    if not task_path.exists():raise RuntimeError('モデル取得失敗: '+repr(errors))
print('HandLandmarker model:',task_path.stat().st_size,'bytes')

## 4. 21点検出：多倍率リトライ・時間的整合性QC

画像フレームごとにIMAGEモードで独立推論し、ROI倍率を複数試して手を検索します（検出率重視）。同じROI中心に近い候補と前フレームの手首位置を優先します。大きな位置ジャンプや左右ラベル切替は**QCとして記録**し、勝手に正しいと判断しません。

- 全36フレーム×21点のCSVを維持。未検出はNaN。
- `handedness_score` は左右分類の確信度であり、関節座標の精度指標ではありません。
- GrabCutマスクは暫定輪郭です。

In [ ]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
LANDMARK_NAMES=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
                'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
                'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
                'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
                'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
EDGES=[(0,1),(1,2),(2,3),(3,4),
       (0,5),(5,6),(6,7),(7,8),
       (0,9),(9,10),(10,11),(11,12),
       (0,13),(13,14),(14,15),(15,16),
       (0,17),(17,18),(18,19),(19,20)]
def roi_crop(frame,box,size=512):
    cx=(box[0]+box[2])/2;cy=(box[1]+box[3])/2
    side=max(box[2]-box[0],box[3]-box[1])*CROP_SCALE
    M=np.float32([[size/side,0,size/2-cx*size/side],
                  [0,size/side,size/2-cy*size/side]])
    patch=cv2.warpAffine(frame,M,(size,size),flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT,borderValue=(114,114,114))
    return patch,np.array([cx,cy],float),float(side)
def make_mask(patch,uv,grabcut=True):
    size=patch.shape[0]
    pts=np.rint(uv).astype(np.int32)
    if not np.isfinite(uv).all():return np.zeros((size,size),np.uint8),'none'
    hull=cv2.convexHull(pts)
    raw=np.zeros((size,size),np.uint8)
    cv2.fillConvexPoly(raw,hull,255)
    # A soft region wider than the skeletal hull approximates hand thickness
    spread=max(9,int(size*.035));kern=np.ones((spread,spread),np.uint8)
    probable=cv2.dilate(raw,kern,iterations=2)
    if not grabcut:return probable,'dilated_hull'
    gc=np.full((size,size),cv2.GC_BGD,dtype=np.uint8)
    gc[probable>0]=cv2.GC_PR_FGD
    gc[raw>0]=cv2.GC_FGD
    # Ensure at least some definite foreground and background for GrabCut
    center=cv2.erode(raw,np.ones((7,7),np.uint8),iterations=1)
    gc[center>0]=cv2.GC_FGD
    if (gc==cv2.GC_FGD).sum()<30 or (gc==cv2.GC_BGD).sum()<30:
        return probable,'dilated_hull_fallback'
    try:
        cv2.grabCut(patch,gc,None,np.zeros((1,65),np.float64),
                    np.zeros((1,65),np.float64),3,cv2.GC_INIT_WITH_MASK)
        mask=np.isin(gc,[cv2.GC_FGD,cv2.GC_PR_FGD]).astype(np.uint8)*255
        if mask.sum()<1000:return probable,'dilated_hull_fallback'
        return mask,'grabcut_provisional'
    except cv2.error:
        return probable,'dilated_hull_fallback'
def mask_to_frame(mask,center,side):
    m=np.float32([[512/side,0,256-center[0]*512/side],
                  [0,512/side,256-center[1]*512/side]])
    inv=cv2.invertAffineTransform(m)
    return cv2.warpAffine(mask,inv,(W,H),flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT,borderValue=0)
def points_to_original(uv,center,side):
    return center[None,:]+(uv-256.)*side/512.
settings=mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task_path)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=HAND_MIN_DET,
    min_hand_presence_confidence=HAND_MIN_PRESENCE)
out_dir=project/OUTPUT_DIR_NAME/video_original.stem
out_dir.mkdir(parents=True,exist_ok=True)
masks_dir=out_dir/'masks_png';masks_dir.mkdir(exist_ok=True)
landmark_rows=[];qc_rows=[];results={}
prev_wrist=None;prev_handedness=None
cap=cv2.VideoCapture(str(video))
if not cap.isOpened():raise RuntimeError('Cannot open video')
with mp_vision.HandLandmarker.create_from_options(settings) as detector:
    for f in tqdm(range(START_FRAME,END_FRAME+1),desc='Multi-scale release hand detection'):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
        ok,frame=cap.read()
        if not ok:raise RuntimeError('Unreadable frame: '+str(f))
        bbox=interp_box(f)
        candidates=[]
        for scale in RETRY_CROP_SCALES:
            original_scale=CROP_SCALE
            CROP_SCALE=float(scale)
            try:
                patch,center,side=roi_crop(frame,bbox)
            finally:
                CROP_SCALE=original_scale
            rgb=np.ascontiguousarray(cv2.cvtColor(patch,cv2.COLOR_BGR2RGB))
            pred=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
            for k,lm in enumerate(pred.hand_landmarks):
                uv=np.array([[q.x*512,q.y*512] for q in lm],dtype=float)
                original=points_to_original(uv,center,side)
                if not np.isfinite(original).all():continue
                visible=((original[:,0]>=0)&(original[:,0]<W)&(original[:,1]>=0)&(original[:,1]<H))
                valid_ratio=float(visible.mean())
                wrist=original[0]
                d_roi=np.linalg.norm(wrist-np.array([(bbox[0]+bbox[2])/2,(bbox[1]+bbox[3])/2]))/max(bbox[2]-bbox[0],bbox[3]-bbox[1],1)
                d_prev=(np.linalg.norm(wrist-prev_wrist)/max(bbox[2]-bbox[0],bbox[3]-bbox[1],1)) if prev_wrist is not None else 0.
                hand_name='';hand_score=np.nan
                if pred.handedness and k<len(pred.handedness) and pred.handedness[k]:
                    hand_name=pred.handedness[k][0].category_name
                    hand_score=float(pred.handedness[k][0].score)
                if valid_ratio>=MIN_LANDMARK_VALID_RATIO and d_roi<=1.8:
                    rank=d_roi+0.45*d_prev+(.025 if hand_name and prev_handedness and hand_name!=prev_handedness else 0)
                    candidates.append((rank,scale,patch,center,side,uv,lm,original,hand_name,hand_score,d_prev))
        candidates.sort(key=lambda z:z[0])
        selected=bool(candidates)
        mask_full=np.zeros((H,W),np.uint8)
        xy=np.full((21,2),np.nan);z=np.full(21,np.nan)
        mask_method='none';handedness='';score=np.nan;used_scale=np.nan;d_prev=np.nan
        jump_flag=0;handedness_switch=0
        if selected:
            rank,used_scale,patch,center,side,uv,lm,xy,handedness,score,d_prev=candidates[0]
            z=np.asarray([p.z for p in lm],float)
            if prev_wrist is not None:
                jump_flag=int(d_prev>MAX_PREV_CENTER_JUMP)
            if prev_handedness and handedness:
                handedness_switch=int(prev_handedness!=handedness)
            m_crop,mask_method=make_mask(patch,uv,RUN_GRABCUT)
            mask_full=mask_to_frame(m_crop,center,side)
            prev_wrist=xy[0].copy()
            prev_handedness=handedness or prev_handedness
        cv2.imwrite(str(masks_dir/f'frame_{f:06d}.png'),mask_full)
        record={'frame':f,'time_s':f/FPS if FPS>0 else np.nan,'hand_detected':int(selected),
           'roi_source':'manual_keyframe' if f in anchor_boxes else 'linear_interpolation',
           'mask_method':mask_method,'mask_area_px':int(np.count_nonzero(mask_full)),
           'handedness_reported':handedness,'handedness_score':score,
           'retry_scale_used':used_scale,'candidate_count':len(candidates),
           'wrist_jump_roi_units':d_prev,'large_jump_flag':jump_flag,
           'handedness_switch_flag':handedness_switch,
           'roi_x0':bbox[0],'roi_y0':bbox[1],'roi_x1':bbox[2],'roi_y1':bbox[3]}
        qc_rows.append(record)
        for j in range(21):
            landmark_rows.append({'frame':f,'time_s':record['time_s'],'landmark_id':j,
               'landmark_name':LANDMARK_NAMES[j],
               'x_px':xy[j,0],'y_px':xy[j,1],'x_norm':xy[j,0]/W,'y_norm':xy[j,1]/H,
               'z_relative':z[j],'detected':int(selected),
               'roi_source':record['roi_source'],'handedness_reported':handedness,
               'large_jump_flag':jump_flag})
        results[f]={'xy':xy,'mask':mask_full,'roi':bbox,'detected':selected,
                    'mask_method':mask_method,'large_jump_flag':jump_flag,
                    'handedness_switch_flag':handedness_switch}
cap.release()
landmarks_csv=out_dir/'hand_21_landmarks_long.csv'
qc_csv=out_dir/'tracking_mask_qc.csv'
pd.DataFrame(landmark_rows).to_csv(landmarks_csv,index=False)
qdf=pd.DataFrame(qc_rows)
qdf.to_csv(qc_csv,index=False)
print('Detected frames:',int(qdf.hand_detected.sum()),'/',len(qdf))
print('QC flagged large jumps:',int(qdf.large_jump_flag.sum()))
print('Handedness switches:',int(qdf.handedness_switch_flag.sum()))
print('Landmarks CSV:',landmarks_csv)

## 5. 骨格＋暫定手マスク：全区間重畳動画・拡大動画

送付候補は **release_zoom_175_210.mp4**。画像に直接「骨格とマスクはAI推定・検証中」と入れます。手が誤検出されたフレームは、数字上の検出成功と区別できるようQCフラグを表示します。

In [ ]:
def annotated_frame(img,f):
    result=results[f]
    a=img.copy();b=np.rint(result['roi']).astype(int)
    cv2.rectangle(a,tuple(b[:2]),tuple(b[2:]),(0,230,0),2)
    mask=result['mask']
    if np.any(mask):
        tint=a.copy();tint[mask>0]=(20,175,20)
        a=cv2.addWeighted(tint,.30,a,.70,0)
        contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(a,contours,-1,(0,255,0),2)
    if result['detected']:
        xy=np.rint(result['xy']).astype(int)
        for i,j in EDGES:
            cv2.line(a,tuple(xy[i]),tuple(xy[j]),(255,255,255),2)
        for i,(x,y) in enumerate(xy):
            cv2.circle(a,(int(x),int(y)),3,(30,30,255),-1)
    label=f"Frame {f} | landmarks={'YES' if result['detected'] else 'NO'} | QC={'CHECK' if result.get('large_jump_flag') or result.get('handedness_switch_flag') else 'OK'}"
    cv2.rectangle(a,(0,0),(min(W,760),39),(0,0,0),-1)
    cv2.putText(a,label,(12,27),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    cv2.putText(a,'AI-estimated landmarks / approximate hand mask (not validated)',(10,H-14),cv2.FONT_HERSHEY_SIMPLEX,.48,(255,255,255),1)
    return a
overlay_path=out_dir/'hand_mask_skeleton_overlay.mp4'
zoom_path=out_dir/f'release_zoom_{START_FRAME}_{END_FRAME}.mp4'
def writer_or_fail(path,dimensions):
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),
                           FPS if FPS>0 else 30,dimensions)
    if not writer.isOpened():raise RuntimeError('Cannot open writer: '+str(path))
    return writer
full_writer=writer_or_fail(overlay_path,(W,H))
zoom_writer=writer_or_fail(zoom_path,(768,768))
cap=cv2.VideoCapture(str(video))
try:
    for f in tqdm(range(START_FRAME,END_FRAME+1),desc='Render overlays'):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
        ok,img=cap.read()
        if not ok:raise RuntimeError('Frame read failed '+str(f))
        overlay=annotated_frame(img,f)
        full_writer.write(overlay)
        b=results[f]['roi']
        cx=(b[0]+b[2])/2;cy=(b[1]+b[3])/2
        side=max(b[2]-b[0],b[3]-b[1])*RELEASE_ZOOM
        affine=np.float32([[768/side,0,384-cx*768/side],[0,768/side,384-cy*768/side]])
        zoom=cv2.warpAffine(overlay,affine,(768,768),borderMode=cv2.BORDER_CONSTANT,
                            borderValue=(25,25,25))
        cv2.putText(zoom,f'Frame {f}',(12,740),cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2)
        zoom_writer.write(zoom)
finally:
    cap.release();full_writer.release();zoom_writer.release()
print('Overlay:',overlay_path)
print('Zoom:',zoom_path)
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,f in zip(axs,np.linspace(START_FRAME,END_FRAME,3).astype(int)):
    ax.imshow(cv2.cvtColor(annotated_frame(frame_at(int(f)),int(f)),cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {f}');ax.axis('off')
plt.show()

## 6. マスクZIP・実行レポート保存とQC

In [ ]:
masks_zip=out_dir/'hand_masks_png.zip'
with zipfile.ZipFile(masks_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for mask_file in sorted(masks_dir.glob('*.png')):
        z.write(mask_file,arcname=mask_file.name)
report={
 'source_video':video_original.name,
 'created_at':datetime.now().isoformat(timespec='seconds'),
 'video_dimensions':[W,H],
 'fps':FPS,
 'frame_range_inclusive':[START_FRAME,END_FRAME],
 'n_frames':END_FRAME-START_FRAME+1,
 'anchor_frames':list(ANCHOR_FRAMES),
 'anchor_boxes':{str(k):v.tolist() for k,v in anchor_boxes.items()},
 'hand_detected_frames':int(qdf.hand_detected.sum()),
 'hand_detection_fraction':float(qdf.hand_detected.mean()),
 'mask_methods':{str(k):int(v) for k,v in qdf.mask_method.value_counts().items()},
 'outputs':{'landmarks_csv':str(landmarks_csv),'qc_csv':str(qc_csv),
            'overlay_mp4':str(overlay_path),'zoom_mp4':str(zoom_path),
            'masks_zip':str(masks_zip)},
 'model':'MediaPipe Hand Landmarker float16',
 'pipeline':'Stage2D independent per-frame multi-scale IMAGE inference + GrabCut',
 'retry_crop_scales':list(RETRY_CROP_SCALES),
 'flagged_wrist_jumps':int(qdf.large_jump_flag.sum()),
 'handedness_switches':int(qdf.handedness_switch_flag.sum()),
 'important_limitations':[
  'The hand mask is an approximate GrabCut/landmark-hull segmentation, not validated semantic segmentation',
  'Landmark x/y are MediaPipe 2D points mapped to original frame, not WiLoR 3D projection',
  'ROI is linearly interpolated between manually chosen boxes, not automatically ground-truth tracked',
  'Missing detections are NaN; no fabricated points or mask',
  'No ball detection or release-frame classification is implemented',
  'Independent per-frame inference does not ensure same-hand identity']
}
report_path=out_dir/'stage2c_report.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Result folder:',out_dir)
plt.figure(figsize=(12,3))
plt.plot(qdf.frame,qdf.hand_detected,drawstyle='steps-mid',label='Detected')
plt.xlabel('Frame');plt.ylabel('Hand detected');plt.ylim(-.1,1.1);plt.grid(True);plt.legend();plt.show()